In [0]:
# ============================================================
# Notebook 6: Batch Deployment
# ATP Tennis Match Winner Prediction
# ============================================================

storage_account = "tennisdatalake60105845"
storage_key = "LjZoIlf5yGSIHxM/9GIXHx5jTq8VNvMOqWjuLZ54krBy3gn+BN7pg8q5/4UM8dgtAEwIMaTNyIYl+AStqzCGpQ=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

# Load the saved model
from pyspark.ml.classification import GBTClassificationModel
from pyspark.ml.feature import VectorAssembler

# Load model from curated zone
model_path = f"abfss://curated@{storage_account}.dfs.core.windows.net/models/gbt_tennis_v1"
gbt_model = GBTClassificationModel.load(model_path)

print(" Model loaded successfully!")
print(f" Model path: {model_path}")

 Model loaded successfully!
 Model path: abfss://curated@tennisdatalake60105845.dfs.core.windows.net/models/gbt_tennis_v1


In [0]:
# ============================================================
# Batch Prediction Job
# ============================================================

# Load the full feature dataset for batch scoring
df_batch = spark.read.format("delta").table("amazon_dbx_60105845.default.atp_tennis_features")

# Define features
feature_cols = [
    "rank_diff", "pts_diff", "odds_diff", "p1_higher_ranked",
    "is_grand_slam", "is_hard_court", "is_clay_court", "is_best_of_5",
    "Rank_1", "Rank_2", "year", "month"
]

# Assemble features
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_assembled = assembler.transform(df_batch)

# Run batch predictions
df_predictions = gbt_model.transform(df_assembled)

# Select output columns
df_output = df_predictions.select(
    "Rank_1", "Rank_2", "rank_diff", "odds_diff",
    "is_grand_slam", "is_hard_court", "is_clay_court",
    "winner_is_player1",
    "prediction"
).withColumnRenamed("winner_is_player1", "actual")

print(f"Batch predictions complete!")
print(f"Total predictions: {df_output.count()}")
df_output.show(10)

Batch predictions complete!
Total predictions: 67433
+------+------+---------+-------------------+-------------+-------------+-------------+------+----------+
|Rank_1|Rank_2|rank_diff|          odds_diff|is_grand_slam|is_hard_court|is_clay_court|actual|prediction|
+------+------+---------+-------------------+-------------+-------------+-------------+------+----------+
|    89|    65|       24| 0.8400000000000001|            0|            1|            0|     0|       0.0|
|     1|     6|       -5|               -0.8|            1|            1|            0|     1|       1.0|
|    58|   112|      -54|               -1.0|            0|            0|            1|     1|       1.0|
|   142|    21|      121|               0.97|            0|            1|            0|     0|       0.0|
|     3|    18|      -15|-3.3200000000000003|            0|            1|            0|     0|       1.0|
|    61|    53|        8|-0.8400000000000001|            0|            0|            1|     0|     

In [0]:
# ============================================================
# Save Batch Predictions to Output Zone
# ============================================================

# Save predictions to processed zone
df_output.write.format("delta") \
    .mode("overwrite") \
    .save(f"abfss://curated@{storage_account}.dfs.core.windows.net/predictions/batch_v1/")

print(" Batch predictions saved to curated zone!")
print(f" Path: abfss://curated@{storage_account}.dfs.core.windows.net/predictions/batch_v1/")

# Quick accuracy check on full dataset
correct = df_output.filter(df_output.actual == df_output.prediction).count()
total = df_output.count()
print(f"\n Full dataset accuracy: {correct/total*100:.2f}%")
print(f" Correct predictions: {correct}/{total}")

 Batch predictions saved to curated zone!
 Path: abfss://curated@tennisdatalake60105845.dfs.core.windows.net/predictions/batch_v1/

 Full dataset accuracy: 69.61%
 Correct predictions: 46943/67433
